# 야생동물 탐지 프로젝트 - 실행 가이드

## 프로젝트 경로
- **WSL (Ubuntu)**: `/home/sherzod/wildlife_detection_project`
- **Windows**: `\\wsl.localhost\Ubuntu\home\sherzod\wildlife_detection_project`

## 📋 전체 프로세스 개요

이 프로젝트는 **2단계 파이프라인**으로 구성됩니다:
1. **YOLO 모델 학습**: 객체 탐지 (야생동물 위치 찾기)
2. **CNN 모델 학습**: 2단계 분류 (오탐 제거 → 종 분류)

> 💡 **참고**: 먼저 사진과 라벨링 작업을 완료해야 합니다. 1단계에서는 이미 라벨링 데이터(JSON)가 있어서 JSON을 YOLO가 인식하도록 변환하는 작업부터 시작합니다.

In [ ]:
---

## 🎯 Part 1: YOLO 모델 학습

### 단계 1-1: JSON → YOLO 데이터셋 변환

**목적**: JSON 형식의 라벨링 데이터를 YOLO 학습용 형식으로 변환

# 1단계: JSON → YOLO 변환
python scripts/build_yolodataset.py \
    --raw_dir raw_data \
    --out_dir output \
    --val_ratio 0.2 \
    --seed 42

# 기본값으로 실행 (가장 간단)
# python scripts/build_yolodataset.py

**이 명령어가 하는 일:**
- `raw_data/`에서 이미지-JSON 쌍 찾기
- JSON의 bbox를 YOLO 형식으로 변환
- train/val로 분할 (기본 80:20)
- `output/dataset/`에 다음 구조 생성:
  - `images/train/`, `images/val/`
  - `labels/train/`, `labels/val/`
  - `data.yaml` (YOLO 학습용 설정 파일)
  - `classes.txt` (클래스 목록)

**옵션 설명:**
- `--raw_dir`: JSON과 이미지가 있는 원본 데이터 디렉토리 (기본값: `raw_data`)
- `--out_dir`: 변환된 YOLO 데이터셋 출력 디렉토리 (기본값: `output`)
- `--val_ratio`: 검증 세트 비율 (기본값: 0.2, 즉 20%)
- `--seed`: 랜덤 시드 (기본값: 42)

---

### 단계 1-2: YOLO 모델 학습

**목적**: 야생동물 객체 탐지 모델 학습 

# 2단계: YOLO 학습
python scripts/train_yolo.py \
    --data output/dataset/data.yaml \
    --model yolov8s.pt \
    --epochs 100 \
    --imgsz 640 \
    --batch 16 \
    --device 0 \
    --project runs/detect \
    --name train

# 기본값으로 실행
# python scripts/train_yolo.py

**이 명령어가 하는 일:**
- YOLOv8 모델을 사용하여 야생동물 탐지 모델 학습
- 학습 결과는 `runs/detect/{name}/weights/`에 저장
  - `best.pt` - 최적 모델
  - `last.pt` - 마지막 에포크 모델

**옵션 설명:**
- `--data`: data.yaml 파일 경로 (기본값: `output/dataset/data.yaml`)
- `--model`: 사전학습 모델 (기본값: `yolov8s.pt`)
- `--epochs`: 에포크 수 (기본값: 100)
- `--imgsz`: 이미지 크기 (기본값: 640)
- `--batch`: 배치 크기 (기본값: 16)
- `--device`: 디바이스 ("0"=GPU 0번, "cpu"=CPU)
- `--project`: 결과 저장 디렉토리 (기본값: `runs/detect`)
- `--name`: 실행 이름 (기본값: `train`)

> ⚠️ **중요**: 1단계를 먼저 실행해야 `output/dataset/data.yaml`이 생성됩니다.  
> 2단계는 1단계가 완료된 후 실행하세요.

---

## 🎯 Part 2: CNN 모델 학습 (2단계 파이프라인)

> 💡 **참고**: CNN 학습은 YOLO 모델이 학습된 후에 이루어져야 합니다.

### 단계 2-1: YOLO 라벨에서 CNN용 Crop 데이터셋 생성

**목적**: YOLO 학습 데이터셋의 bbox를 crop하여 CNN 학습용 이미지 생성

In [ ]:
# 1단계: YOLO crop 데이터셋 생성
python scripts/build_cnn_crops.py \
    --config config/cnn_stages.yaml \
    --padding 0.1

# 기본값으로 실행
# python scripts/build_cnn_crops.py

**이 단계가 하는 일:**
- `output/dataset/images/train`, `output/dataset/images/val`의 이미지와 라벨 읽기
- YOLO 라벨의 bbox를 crop (padding 10% 적용)
- **Stage1용**: 모든 동물 crop을 `cnn_dataset/stage1_animal_vs_background/train/animal/`, `val/animal/`에 저장
- **Stage2용**: 클래스별로 `cnn_dataset/stage2_species/train/<클래스명>/`, `val/<클래스명>/`에 저장
  - 8종: 개, 고라니, 고양이, 너구리, 노루, 멧돼지, 조류, 족제비

**옵션:**
- `--config`: 설정 파일 경로 (기본값: `config/cnn_stages.yaml`)
- `--padding`: bbox 확장 비율 (기본값: 0.1 = 10%)

---

### 단계 2-2: non_animal(오탐) 샘플 수집

**목적**: Stage1 CNN 학습을 위한 오탐 샘플 수집

In [ ]:
# 2단계: non_animal 샘플 수집
python scripts/collect_non_animal_crops.py \
    --config config/cnn_stages.yaml \
    --model models/best.pt \
    --conf 0.25 \
    --iou 0.3 \
    --max-per-image 5 \
    --padding 0.1

# 기본값으로 실행
# python scripts/collect_non_animal_crops.py

**이 단계가 하는 일:**
- 학습된 YOLO 모델로 train/val 이미지 추론
- GT와 IoU가 낮은 예측을 오탐으로 간주
- 오탐 bbox를 crop하여 `cnn_dataset/stage1_animal_vs_background/train/non_animal/`, `val/non_animal/`에 저장

**옵션:**
- `--config`: 설정 파일 경로
- `--model`: YOLO 모델 경로 (비우면 config의 `yolo_model_pt` 사용)
- `--conf`: 오탐 후보로 쓸 예측 최소 confidence (기본값: 0.25)
- `--iou`: GT와 IoU가 이 값 미만이면 오탐으로 간주 (기본값: 0.3)
- `--max-per-image`: 이미지당 최대 non_animal crop 수 (기본값: 5)
- `--padding`: crop bbox 확장 비율 (기본값: 0.1)

> ⚠️ **중요**: 이 단계는 YOLO 모델이 학습되어 있어야 합니다 (`models/best.pt` 또는 `runs/detect/.../weights/best.pt`)

---

### 단계 2-3: Stage 1 CNN 학습 (animal vs non_animal)

**목적**: 오탐 제거를 위한 이진 분류 모델 학습

In [ ]:
# 3단계: Stage 1 CNN 학습 (animal vs non_animal)
python scripts/train_cnn.py \
    --stage 1 \
    --config config/cnn_stages.yaml \
    --epochs 20 \
    --batch 32 \
    --img-size 224 \
    --lr 1e-4

**이 단계가 하는 일:**
- MobileNetV2 기반 이진 분류 모델 학습
- 입력: `cnn_dataset/stage1_animal_vs_background/train/` (animal/, non_animal/)
- 출력: `runs/cnn/stage1/best.keras` (최적 모델), `final.keras`, `class_names.json`

**옵션:**
- `--stage`: 1 (필수)
- `--epochs`: 에포크 수 (기본값: 20)
- `--batch`: 배치 크기 (기본값: 32)
- `--img-size`: 입력 이미지 크기 (기본값: 224)
- `--lr`: 학습률 (기본값: 1e-4)

---

### 단계 2-4: Stage 2 CNN 학습 (8종 분류)

**목적**: 동물 종 분류를 위한 다중 분류 모델 학습

In [ ]:
# 4단계: Stage 2 CNN 학습 (8종 분류)
python scripts/train_cnn.py \
    --stage 2 \
    --config config/cnn_stages.yaml \
    --epochs 30 \
    --batch 32 \
    --img-size 224 \
    --lr 1e-4

**이 단계가 하는 일:**
- MobileNetV2 기반 다중 분류 모델 학습
- 입력: `cnn_dataset/stage2_species/train/` (8개 클래스 폴더)
- 출력: `runs/cnn/stage2/best.keras` (최적 모델), `final.keras`, `class_names.json`

**옵션:**
- `--stage`: 2 (필수)
- `--epochs`: 에포크 수 (기본값: 30)
- `--batch`: 배치 크기 (기본값: 32)
- `--img-size`: 입력 이미지 크기 (기본값: 224)
- `--lr`: 학습률 (기본값: 1e-4)

---

## 📁 학습된 모델 파일 위치

### YOLO 모델
- **위치**: `models/best.pt` 또는 `runs/detect/{name}/weights/best.pt`
- **용도**: 야생동물 객체 탐지

### CNN 모델
- **Stage 1**: `runs/cnn/stage1/best.keras` - 오탐 제거 모델 (animal vs non_animal)
- **Stage 2**: `runs/cnn/stage2/best.keras` - 종 분류 모델 (8종 분류)

---

## 🚀 전체 파이프라인 실행 순서 (한 번에)

```bash
# Part 1: YOLO 학습
python scripts/build_yolodataset.py      # JSON → YOLO 변환
python scripts/train_yolo.py              # YOLO 학습

# Part 2: CNN 학습
python scripts/build_cnn_crops.py        # YOLO crop 데이터셋 생성
python scripts/collect_non_animal_crops.py  # non_animal 샘플 수집
python scripts/train_cnn.py --stage 1    # Stage 1 CNN 학습
python scripts/train_cnn.py --stage 2    # Stage 2 CNN 학습
```

**중요 사항:**
- 각 단계는 순서대로 실행해야 합니다
- 이전 단계의 결과물이 다음 단계의 입력으로 사용됩니다
- CNN 학습은 YOLO 모델이 학습된 후에 진행해야 합니다

---

## 📦 모델 파일 복사

학습된 모델 파일들을 "결과물" 폴더에 복사합니다.